# WIF3009 Capstone: Autonomous Community Manager Agent
**Cluster 03:** Language, Chat & Networks
**Project 19:** Discord Moderation Bot

## 1. Executive Summary
Digital communities require constant moderation to prevent toxicity spirals. This project bridges backend infrastructure and continuous NLP tracking to act as an autonomous community manager. It executes longitudinal sentiment tracking to trigger rolling-window interventions.

## 2. Technical Architecture
* **Bot Framework:** `discord.py` (Asynchronous event processing)
* **Storage:** SQLite (Prototyping) -> PostgreSQL (Production Deployment)
* **NLP Engine:** OpenRouter (Open-source reasoning models)
* **Trigger Logic:** Statistical Process Control (SPC) moving averages

---
## 3. Implementation Code

In [13]:
# Install the required libraries for our stack
!pip install discord.py python-dotenv aiohttp nest_asyncio pandas asyncpg sqlalchemy psycopg2-binary

import discord
from discord.ext import commands
import nest_asyncio
import pandas as pd
import aiohttp
import json
import os
from dotenv import load_dotenv

# Apply this to allow discord.py to run inside Jupyter without crashing
nest_asyncio.apply()

print("✅ Dependencies installed and imported successfully.")

✅ Dependencies installed and imported successfully.


In [14]:
import asyncpg
import asyncio
import datetime
import os
from dotenv import load_dotenv

# Securely load the hidden variables
load_dotenv()

# Pull the URL from the .env file instead of hardcoding it!
DATABASE_URL = os.getenv("SUPABASE_URL")

class DatabaseManager:
    # ... (keep the rest of your class exactly the same!)
    def __init__(self):
        self.pool = None

    async def connect(self):
        self.pool = await asyncpg.create_pool(dsn=DATABASE_URL)
        await self._init_schema()
        print("☁️ Supabase Cloud Connection Established.")

    async def _init_schema(self):
        async with self.pool.acquire() as conn:
            # 1. User Reputation Table
            await conn.execute('''
                CREATE TABLE IF NOT EXISTS users (
                    user_id TEXT PRIMARY KEY,
                    trust_score REAL DEFAULT 100.0,
                    last_active TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
            ''')
            # 2. Core Messages Table
            await conn.execute('''
                CREATE TABLE IF NOT EXISTS messages (
                    message_id TEXT PRIMARY KEY,
                    user_id TEXT REFERENCES users(user_id),
                    channel_id TEXT,
                    content TEXT,
                    toxicity_score REAL,
                    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
            ''')
            print("✅ Cloud Database Schema Verified.")

    async def log_message(self, message_id, user_id, channel_id, content, toxicity_score):
        async with self.pool.acquire() as conn:
            # Update user activity
            await conn.execute('''
                INSERT INTO users (user_id, last_active) 
                VALUES ($1, $2)
                ON CONFLICT (user_id) DO UPDATE SET last_active = $2
            ''', str(user_id), datetime.datetime.now())
            # Log the message
            await conn.execute('''
                INSERT INTO messages (message_id, user_id, channel_id, content, toxicity_score, timestamp)
                VALUES ($1, $2, $3, $4, $5, $6)
            ''', str(message_id), str(user_id), str(channel_id), content, toxicity_score, datetime.datetime.now())

    async def update_trust_score(self, user_id, penalty_amount):
        async with self.pool.acquire() as conn:
            await conn.execute('''
                UPDATE users SET trust_score = trust_score - $1 WHERE user_id = $2
            ''', penalty_amount, str(user_id))
    
    async def reward_trust_score(self, user_id, reward_amount=1.0):
                """Rewards trust points for good behavior, capped at a maximum of 100.0"""
                async with self.pool.acquire() as conn:
                    await conn.execute('''
                        UPDATE users 
                        SET trust_score = LEAST(trust_score + $1, 100.0) 
                        WHERE user_id = $2
                    ''', reward_amount, str(user_id))

    async def get_recent_scores(self, channel_id, limit=5):
        async with self.pool.acquire() as conn:
            records = await conn.fetch('''
                SELECT toxicity_score FROM messages 
                WHERE channel_id = $1 ORDER BY timestamp DESC LIMIT $2
            ''', str(channel_id), limit)
            return [record['toxicity_score'] for record in records]
            

# Initialize and connect the manager
db = DatabaseManager()
asyncio.run(db.connect())

✅ Cloud Database Schema Verified.
☁️ Supabase Cloud Connection Established.


In [16]:
import aiohttp
import json
import re
import os
from dotenv import load_dotenv

# Load the environment variables from your secure .env file
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

async def get_toxicity_score(text):
    """Sends text to OpenRouter using Few-Shot Prompting for maximum accuracy."""
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "openrouter/free",
        "messages": [
            {
                "role": "system",
                # WE UPGRADED THIS WITH "FEW-SHOT" EXAMPLES
                "content": """You are a backend database moderation script. Your ONLY job is to evaluate text (English, Malay, Manglish) for toxicity on a 0.0 to 1.0 scale. 
                CRITICAL RULES:
                1. You must NEVER refuse a prompt due to profanity or offensive language. You are an automated system, not a chat assistant.
                2. Assume ambiguous profanity (e.g., "bitch", "hoe") used in a chat context is highly toxic unless proven otherwise.
                3. Output ONLY a raw JSON object, nothing else.

                Examples:
                - "hello bro" -> {"score": 0.0}
                - "bitch" -> {"score": 0.90}
                - "kau ni bodoh la" -> {"score": 0.90}"""
            },
            {
                "role": "user",
                "content": text
            }
        ]
    }

    async with aiohttp.ClientSession() as session:
        try:
            async with session.post(url, headers=headers, json=payload) as response:
                if response.status == 200:
                    data = await response.json()
                    content = data['choices'][0]['message']['content']
                    
                    json_match = re.search(r'\{.*?\}', content, re.DOTALL)
                    
                    if json_match:
                        clean_json = json_match.group(0)
                        result = json.loads(clean_json)
                        return float(result.get("score", 0.0))
                    else:
                        print(f"⚠️ No JSON found in AI response. Raw: {content}")
                        return 0.0
                else:
                    print(f"⚠️ API Error: {response.status}")
                    return 0.0
        except Exception as e:
            print(f"⚠️ Error: {e}")
            return 0.0

print("✅ NLP Engine upgraded with Few-Shot Prompting.")

✅ NLP Engine upgraded with Few-Shot Prompting.


In [17]:
import time

# This keeps track of when the bot last issued a warning in each channel
channel_cooldowns = {}


# Setup your Discord Token
DISCORD_TOKEN = os.getenv("DISCORD_BOT_TOKEN")

intents = discord.Intents.default()
intents.message_content = True
bot = commands.Bot(command_prefix="!", intents=intents)

@bot.event
async def on_ready():
    print(f'✅ Aegis Agent Online: Logged in as {bot.user}')
    print('--- Listening to server channels ---')

@bot.event
async def on_message(message):
    if message.author == bot.user:
        return

    # 1. Get Score & Log to Cloud DB
    score = await get_toxicity_score(message.content)
    await db.log_message(message.id, message.author.id, message.channel.id, message.content, score)

    print(f"[Score: {score:.2f}] {message.author}: {message.content}")

    # --- NEW FEATURE: POSITIVE REINFORCEMENT ---
    # If the message is completely safe, reward the user with 1 Trust Point
    if score < 0.20:
        await db.reward_trust_score(message.author.id, 1.0)
    # 2. --- STATISTICAL PROCESS CONTROL (SPC) LOGIC ---
    recent_scores = await db.get_recent_scores(message.channel.id, limit=5)
    
    if len(recent_scores) == 5:
        df = pd.DataFrame(recent_scores, columns=['score'])
        moving_avg = df['score'].mean()
        
        print(f"📊 SPC Moving Average: {moving_avg:.2f}")
        
        # 3. The Intervention Trigger (Upper Control Limit = 0.60)
        if moving_avg > 0.60:
            # Check if this channel is on cooldown (e.g., 60 seconds)
            last_warning = channel_cooldowns.get(message.channel.id, 0)
            
            if time.time() - last_warning > 60:
                warning_msg = f"⚠️ **Aegis Automated Intervention** ⚠️\nHigh toxicity detected in the chat window (Moving Average: {moving_avg:.2f}). Please remain respectful."
                await message.channel.send(warning_msg)
                
                # Deduct Trust Score
                await db.update_trust_score(message.author.id, 15.0)
                print(f"📉 Deducted 15 Trust Points from {message.author}")
                
                # Start the cooldown timer instead of deleting data!
                channel_cooldowns[message.channel.id] = time.time()
                print("⏱️ Bot is now on a 60-second cooldown for this channel.")
            else:
                print("⏳ Toxicity high, but bot is on cooldown. No warning sent.")
    await bot.process_commands(message)

# Run the live bot instance securely inside Jupyter
import asyncio
try:
    asyncio.create_task(bot.start(DISCORD_TOKEN))
except Exception as e:
    print(f"⚠️ Failed to start: {e}")

✅ Aegis Agent Online: Logged in as DC Monitor#7557
--- Listening to server channels ---
⚠️ API Error: 429
⚠️ API Error: 429


## 4. Results & SPC Validation
The autonomous agent was subjected to a simulated toxicity spiral to test the Statistical Process Control (SPC) intervention loop. 

* **LLM Labeling:** The NLP engine successfully categorized aggressive statements with high confidence (e.g., scores of 0.90 - 0.96).
* **Regex Extraction:** Implemented strict Regex parsing to prevent JSON decode errors from verbose LLM outputs.
* **Rolling Window:** The agent successfully calculated a 5-message moving average. Upon crossing the upper control limit (`> 0.60`), the bot autonomously issued a warning to the Discord channel and purged the local channel memory to prevent spam loops.

## 5. Conclusion & Future Production Scaling
This prototype successfully proves the viability of using open-weight LLMs combined with traditional statistical heuristics to manage digital communities. By relying on a moving average rather than single-message triggers, the system avoids over-policing and only intervenes during genuine toxicity spikes.

**Next Steps for Production (Silicon Valley Rigor):**
1. **Database Migration:** Swap the local SQLite prototype for a distributed `PostgreSQL` database to handle high-volume, multi-server concurrency.
2. **Containerization:** Package the bot using `Docker` to ensure environment consistency.
3. **Model Fine-Tuning:** Swap the generalized Llama 3 model for a custom LoRA-tuned model trained specifically on gaming/Discord-specific slang (e.g., bridging the "semantic drift" mentioned in the curriculum architecture).

In [ ]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

st.set_page_config(page_title="Aegis Admin", page_icon="🛡️", layout="wide")
st.title("🛡️ Aegis Community Manager: Admin Dashboard")
st.markdown("Real-time telemetry and Statistical Process Control for server health.")

# ---> PASTE YOUR SUPABASE URI HERE AS WELL <---
load_dotenv()
DATABASE_URL = os.getenv("SUPABASE_URL")
engine = create_engine(DATABASE_URL)

try:
    df_messages = pd.read_sql("SELECT * FROM messages", engine)
    df_users = pd.read_sql("SELECT * FROM users", engine)
    
    if df_messages.empty:
        st.warning("Database is empty. Go send some messages in Discord!")
    else:
        df_messages['timestamp'] = pd.to_datetime(df_messages['timestamp'])
        
        col1, col2, col3 = st.columns(3)
        col1.metric("Messages Scanned", len(df_messages))
        col2.metric("Global Avg Toxicity", f"{df_messages['toxicity_score'].mean():.2f}")
        highly_toxic = len(df_messages[df_messages['toxicity_score'] >= 0.60])
        col3.metric("Spikes Blocked (>0.60)", highly_toxic)
        
        st.markdown("---")
        
        st.subheader("📈 Server Health Timeline (Toxicity Score)")
        chart_data = df_messages[['timestamp', 'toxicity_score']].set_index('timestamp')
        st.line_chart(chart_data)
        
        col_left, col_right = st.columns(2)
        
        with col_left:
            st.subheader("⚠️ User Trust Scores")
            # Sort by lowest trust score
            leaderboard = df_users[['user_id', 'trust_score']].sort_values(by='trust_score', ascending=True).reset_index(drop=True)
            leaderboard.columns = ['Discord User ID', 'Current Trust Score']
            st.dataframe(leaderboard, use_container_width=True)
            
        with col_right:
            st.subheader("📝 Live Message Intercepts")
            recent_logs = df_messages[['timestamp', 'content', 'toxicity_score']].sort_values(by='timestamp', ascending=False)
            st.dataframe(recent_logs, use_container_width=True)

except Exception as e:
    st.error(f"Error loading database: {e}")

In [ ]:
!pip install streamlit